# Day 18 — Solution: Which SE When

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_multi(X_cols, y):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in X_cols])
    y = np.asarray(y, float)
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - k)
    se_c = np.sqrt(np.diag(s2 * XtX_inv))
    return {"b": b, "se_c": se_c, "resid": e, "X": X, "XtX_inv": XtX_inv}

def se_white(X, e):
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = X.T @ (e[:, None] ** 2 * X)
    k = X.shape[1]; n = X.shape[0]
    return np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv)) * np.sqrt(n / (n - k))

def se_nw(X, e, L):
    xe = X * e[:, None]
    meat = xe.T @ xe
    for l in range(1, L + 1):
        w = 1 - l / (L + 1)
        G = xe[l:].T @ xe[:-l]
        meat += w * (G + G.T)
    XtX_inv = np.linalg.inv(X.T @ X)
    return np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv))

## E1 — the worlds

In [ ]:
def w_iid(rng, n=1200):
    m = rng.normal(0, 0.011, n)
    y = 0.0 * m + rng.normal(0, 0.010, n)
    return m, y

def w_het(rng, n=1200):
    m = rng.normal(0, 0.011, n)
    sig = 0.006 * (0.4 + 3.5 * np.abs(m) / 0.011)
    y = 0 + 0.0 * m + rng.normal(0, 1, n) * sig     # beta = 0 exactly
    return m, y

def w_overlap(rng, n=3000):
    r = rng.normal(0, 0.01, n)
    x = pd.Series(r).rolling(21).sum().shift(1).values
    y = pd.Series(r).rolling(21).sum().shift(-21).values  # 0 * x + fresh noise sums
    keep = ~(np.isnan(x) | np.isnan(y))
    return x[keep], y[keep]

def w_panel(rng, N=50, T=120):
    m_t = rng.normal(0, 1, T)
    betas = rng.uniform(0.5, 1.5, N)
    R = np.outer(m_t, betas) + rng.normal(0, 1.0, (T, N))
    z = rng.normal(0, 1, N)                    # persistent per name, true effect 0
    return R, z

rng = np.random.default_rng(1)
mx, my = w_het(rng)
f = ols_multi([mx], my)
ev = pd.Series(f["resid"])
print(f"het check: var(e) by |x| half: {ev[np.abs(mx)<np.median(np.abs(mx))].var():.4f} vs {ev[np.abs(mx)>=np.median(np.abs(mx))].var():.4f}")
xo, yo = w_overlap(rng)
print(f"overlap check: ACF(e_signed proxy)=ACF(y): {[f'{pd.Series(yo).autocorr(l):+.2f}' for l in [1,10,21]]}")
R, Z = w_panel(rng)
print(f"panel check: mean cross-correlation of residuals ~= common factor share: {np.corrcoef(R.T)[np.triu_indices(50,1)].mean():+.2f}")

**Expected reasoning.** Diagnostics confirm the planted structure on one
draw each: variance ratio ≈ 1:6 across |x| halves (het); the ramp ACF (0.95,
0.52, ≈0) (overlap); cross-correlation ≈ 0.2–0.3 after the idio noise (panel
— modest but ruinous at N = 50). The effective-n remark from the exercise:
the overlap world's 3000 overlapped "months" are ≈ 143 independent ones —
**which is why its classical t's are inflated by ~√21, and why power
discussions for long-horizon regressions must use effective n.**

## E2 — the tournament

In [ ]:
n_runs = 300
rej = {w: {e: 0 for e in ["classical", "white", "nw3", "nw21", "fm"]} for w in
       ["iid", "het", "overlap", "panel"]}
for i in range(n_runs):
    rng = np.random.default_rng(40000 + i)

    x, y = w_iid(rng); f = ols_multi([x], y)
    rej["iid"]["classical"] += abs(f["b"][1]/f["se_c"][1]) > 1.96
    rej["iid"]["white"] += abs(f["b"][1]/se_white(f["X"], f["resid"])[1]) > 1.96
    rej["iid"]["nw3"] += abs(f["b"][1]/se_nw(f["X"], f["resid"], 3)[1]) > 1.96

    x, y = w_het(rng); f = ols_multi([x], y)
    # test the FALSE-beta null correctly: truth is beta=0 here; t on coefficient
    rej["het"]["classical"] += abs(f["b"][1]/f["se_c"][1]) > 1.96
    rej["het"]["white"] += abs(f["b"][1]/se_white(f["X"], f["resid"])[1]) > 1.96

    x, y = w_overlap(rng); f = ols_multi([x], y)
    rej["overlap"]["classical"] += abs(f["b"][1]/f["se_c"][1]) > 1.96
    rej["overlap"]["white"] += abs(f["b"][1]/se_white(f["X"], f["resid"])[1]) > 1.96
    rej["overlap"]["nw3"] += abs(f["b"][1]/se_nw(f["X"], f["resid"], 3)[1]) > 1.96
    rej["overlap"]["nw21"] += abs(f["b"][1]/se_nw(f["X"], f["resid"], 21)[1]) > 1.96

    R, z = w_panel(rng)
    fp = ols_multi([np.tile(z, R.shape[0])], R.ravel())
    rej["panel"]["classical"] += abs(fp["b"][1]/fp["se_c"][1]) > 1.96
    lams = np.array([ols_multi([z], R[t])["b"][1] for t in range(R.shape[0])])
    rej["panel"]["fm"] += abs(lams.mean()/(lams.std(ddof=1)/np.sqrt(len(lams)))) > 1.96

tab = pd.DataFrame(rej).T / n_runs
print((tab * 100).round(1).astype(str).replace("0.0", "·").to_string())

**Expected matrix (percent, nominal 5).**

| | classical | white | nw3 | nw21 | fm |
|---|---|---|---|---|---|
| iid | ≈4–5 | ≈4–5 | ≈5 | — | — |
| het | ≈23 | ≈4–5 | — | — | — |
| overlap | ≈60–65 | ≈60–65 | ≈28 | ≈11–14 | — |
| panel | ≈35–40 | — | — | — | ≈4 |

Every number is a receipt from days 4, 5, 8, 15 — now side by side: classical
is honest in exactly ONE world; each sandwich repairs exactly its own
mechanism; NW(3)-on-overlap is the classic false comfort; and NW(21) repairs
most but not all of the overlap damage (≈11–14%: HAC is *consistent*, not
exact — at near-unit-root autocorrelation it under-covers at this n; the
exact answer remains the non-overlapped re-estimate, day 8's bottom line).
FM stands alone in the panel row.

## E3 — reading the table (exemplar)

(a) The honest sets per world: iid — all (~5%; robust flavors cost
nothing measurable at n = 1200 — the sd of the SE estimate itself runs a
few % higher for White/NW, an affordable tax, but a tax that grows in
smaller samples); het — White only (classical fires ~20%, a 4× lie);
overlap — NW(21) only (classical ~2-in-3 false positives; White is
classical in disguise here — diagonal meat, memoryless world; NW(3)
repairs half the mechanism and leaves ~25% — partial repair is still
failure); panel — FM only (pooled classical ~35–40%). (e) The iid-row
lesson repeated for your future reviewers: over-robustifying in a healthy
world costs ~nothing at research n, while under-robustifying outside is
catastrophic — hence the field's robust-by-default norm.

## E4 — the flowchart (exemplar)

```
Is the observation structure a panel (units sharing periods)?
├─ YES → per-period cross-sections, average slopes: Fama–MacBeth.
│        Check slope-series ACF; if |ρ| > ~0.15, HAC the slope series.
│        (Module 09 upgrade: two-way cluster when cross- AND time-
│        dependence persist within the pooled residuals.)
└─ NO (single series) →
   ├─ Overlapping windows or persistent+autocorrelated residuals?
   │   → Newey–West, L ≥ mechanism horizon (ramp ACF tells you).
   ├─ Fan / time-bucket variance varies (no autocorrelation)?
   │   → White/HC1.
   └─ Neither → classical. (Report robust-in-footnote only if the
      audience needs the reassurance; the honest word is "classical,
      verified".)
```

Past exercises mapped: day-1/3 planted worlds → bottom branch; day-4(b) →
White branch; day-8 overlap — NW branch; day-14 per-name regressions →
White branch (their panel synthesis was already step-2 FM logic); day-15/16
→ top branch with the ACF caveat.

## E5 — the stranger's table (exemplar)

(i) Pooled firm-months with "robust": suspected White; structure demands
FM (or cluster-by-date). Overlap world's lesson doesn't apply, but the
panel row does: haircut t by 2–3× → t ≈ 1.7–2.6. Confidence: **suspect**;
also demand the λ̂_t share-positive row before believing steadiness.

(ii) Daily-overlapped 63-day trend windows, classical: demands NW ≥ 63.
Expected repair: t/√(63) ≈ t/7.9 worst case; plausibly t ≈ 0.6–1.2.
Confidence: **gone** — the finding IS the overlap; the non-overlapped
re-estimate (every-63rd-day) is the honesty test, and the paper must show
it.

(iii) Single-name daily, White, t = 3.1 for β ≠ 1: the structure is a
plain time series — White's own reach is correct for the fan, but the
stated TEST (t on β − 1, not on β) is a stronger claim needing the same
SE — fine — and the persistent-regressor worry is nil (market returns are
nearly iid). Confidence: **mostly intact**; residual checks: crisis-window
stability (day 12's δ̂) is the meaningful next question, not the SE
flavor.

**The day's sentence to keep:** the standard error is a *claim about the
residual process you could not fully observe* — and the tournament says
what each flavor of that claim is worth when you're wrong.